In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
import requests
import pandas as pd
import io
import zipfile

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
url = "https://www.nrscotland.gov.uk/media/3zjlgpt3/sspl_2025_1.zip"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print(z.namelist())

In [0]:
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("SingleRecord.csv") as f:
        df_pd = pd.read_csv(f)

print(df_pd.shape)
print(df_pd.columns.tolist())
print(df_pd.head())

In [0]:
columnts_to_keep = [
    "Postcode",
    "DateOfDeletion",
    "CouncilArea2019Code",
    "DataZone2022Code",
    "UrbanRural6Fold2022Code",
    "ScottishIndexOfMultipleDeprivation2020Rank"
]

df_pd_slim = df_pd[columnts_to_keep]

df_pd_active = df_pd_slim[df_pd_slim["DateOfDeletion"].isna()]

print(df_pd_active.shape)
print(df_pd_active.head(3))

In [0]:
df_pd_final = df_pd_active.drop(columns=["DateOfDeletion"])
df_bronze = spark.createDataFrame(df_pd_final)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.postcode_lookup")
)

In [0]:
#sanity check
spark.read.table("bronze.postcode_lookup").show(5, truncate=False)
spark.read.table("bronze.postcode_lookup").count()